## 引言 Introduction

通用物体检测需快速、准确且能识别多种物体，但现有检测数据集规模远小于分类数据集，限制了检测范围。为此，研究人员提出分层对象分类视角与联合训练算法，可融合检测和分类数据；先改进基础 YOLO 系统得到先进实时探测器 YOLOv2，再通过该方法在ImageNet的9000个类别和COCO数据集上训练出能检测 9000 余种物体类别的实时探测器 YOLO9000，其代码和预训练模型可在指定网站获取。

## 更好 Better

YOLO相对于Fast R-CNN召回率低和定位不准的问题。
- 没有使用过分增加网络深度，具体的改进尝试见下表：
    |                | YOLO  |      |      |      |      |      |      |      | YOLOv2 |
    |----------------|-------|------|------|------|------|------|------|------|--------|
    | batch norm?    |       | ✓    | ✓    | ✓    | ✓    | ✓    | ✓    | ✓    | ✓      |
    | hi-res classifier? |    |      | ✓    | ✓    | ✓    | ✓    | ✓    | ✓    | ✓      |
    | convolutional? |       |      |      | ✓    | ✓    | ✓    | ✓    | ✓    | ✓      |
    | anchor boxes?  |       |      |      | ✓    | ✓    |      |      |      |        |
    | new network?   |       |      |      |      | ✓    | ✓    | ✓    | ✓    | ✓      |
    | dimension priors? |    |      |      |      |      | ✓    | ✓    | ✓    | ✓      |
    | location prediction? |  |      |      |      |      | ✓    | ✓    | ✓    | ✓      |
    | passthrough?   |       |      |      |      |      |      | ✓    | ✓    | ✓      |
    | multi-scale?   |       |      |      |      |      |      |      | ✓    | ✓      |
    | hi-res detector? |     |      |      |      |      |      |      |      | ✓      |
    |----------------|-------|------|------|------|------|------|------|------|--------|
    | VOC2007 mAP    | 63.4  | 65.8 | 69.5 | 69.2 | 69.6 | 74.4 | 75.4 | 76.8 | **78.6** |
- **批归一化 Batch Normalization**: 在每个卷积层后添加BN层（避免过拟合同时移除dropout），提升`2%` **mAP**
- **高分辨率分类器 High Resolution Classifier**: 
  - YOLO1使用(224 x 224)训练模型并提升到448用于检测。
  - YOLO2使用(448 x 448)在ImageNet上微调10个epochs。
  - 提升`4%` **mAP**
- **带锚点的卷积层 Convolutional With Anchor Boxes**: 
  - YOLO1直接通过顶部的全连通层预测边界框的坐标(x, y, w, h, confidence)
  - YOLO2(模仿Faster R-CNN)使用预定义锚框预测类(class: con prob)和对象性(objectness: IOU)
  - 准确率: 69.5% -> 69.2% mAP; 召回率: 81% -> 88%
- **维度簇 Dimension Clusters**(第一个问题): 
  - 使用k-means聚类算法。传统的欧式距离会放大大框的影响，改用:$d(\text{box}, \text{centroid}) = 1 - \text{IOU}(\text{box}, \text{centroid})$作为距离度量。
  - <image src="./assets/clusters2.png" style="width: 30%;"/>
- **直接定位预测 Direct location prediction**(第二个问题): 

## Strong

一、核心问题与解决方案框架
- 现有通用物体检测面临两大关键限制：
    - 一是检测数据集规模远小于分类数据集（如 COCO 仅含数十类检测标签，ImageNet 含数万分类标签），导致可检测类别有限；
    - 二是数据集标签结构不兼容（检测标签多为通用类别如 “狗”，分类标签含细分类别如 “诺福克梗”，传统 softmax 假设类别互斥，无法直接融合）。

为此，研究团队提出 “WordTree 层级分类模型 + 联合训练机制” 的组合方案，核心目标是利用海量分类数据扩展检测模型的类别覆盖范围，同时保留检测任务所需的定位能力。

二、关键技术细节
1. WordTree 层级分类模型：解决数据集融合难题
核心思路：基于 WordNet（语言概念数据库）构建视觉概念层级树，将不同数据集的非互斥标签映射到树的不同节点，实现标签体系的统一。
构建过程：
以 “物理对象” 为根节点，提取 ImageNet 等数据集中视觉名词在 WordNet 中的路径；
优先保留单一路径的概念，对多路径概念选择最短路径（最小化树结构复杂度），最终形成层级树（如 “诺福克梗”→“梗犬”→“猎犬”→“狗”→“犬科”→“物理对象”）。
分类逻辑：
模型在每个节点预测 “子类别相对于父类别” 的条件概率（如 P (诺福克梗 | 梗犬)）；
某类别的绝对概率通过路径上所有条件概率相乘得到（如 P (诺福克梗)=P (诺福克梗 | 梗犬)×P (梗犬 | 猎犬)×…×P (物理对象)=1）；
替代传统单一 softmax，对每个父节点下的子节点独立计算 softmax，兼容 “通用类别 - 细分类别” 的非互斥关系。
1. 联合训练机制：融合检测与分类数据
数据混合策略：将检测数据集（如 COCO，含边界框 + 类别标签）与分类数据集（如 ImageNet，仅含类别标签）混合输入模型，按数据类型差异化计算损失：
输入检测图像时：采用完整 YOLOv2 损失函数，反向传播 “边界框坐标预测”“物体性判断”“类别分类” 三类损失；
输入分类图像时：仅反向传播分类相关损失，同时假设模型预测的最高置信度边界框与真实目标的 IOU≥0.3，反向传播物体性损失。
标签处理规则：
检测图像标签（如 “狗”）：仅在标签对应节点及上层节点计算分类损失，不向下层细分节点（如 “德国牧羊犬”）分配误差（因无细分标注）；
分类图像标签（如 “诺福克梗”）：沿层级树向上传播标签，同时在所有对应节点计算分类损失。
1. YOLO9000 模型构建与训练
基础架构：基于 YOLOv2 优化，将先验框数量从 5 个减至 3 个，控制输出维度以适配 9000 + 类别的预测需求；
数据集配置：
融合 COCO 检测数据集与 ImageNet top9000 分类数据集，补充 ImageNet 检测挑战中未覆盖的类别；
最终 WordTree 含 9418 个类别，通过过采样 COCO 平衡数据量（ImageNet 与 COCO 比例为 4:1）；
训练目标：通过检测数据学习 “精确定位物体” 的能力，通过分类数据扩展 “识别海量类别” 的词汇量与鲁棒性。
三、模型性能与结果分析
1. 核心性能指标
检测范围：可实时识别 9000 余种物体类别；
ImageNet 检测任务表现：整体 mAP 达 19.7，其中 156 个 “仅见过分类数据、未见过检测标注” 的类别 mAP 达 16.0，优于传统 DPM 模型；
层级分类验证：在 ImageNet 1000 类数据集构建的 WordTree1k（含 1369 个节点）上，Darknet-19 模型仍保持 71.9% top-1 准确率、90.4% top-5 准确率，仅较扁平分类略有下降。
1. 优势与局限
优势：首次实现 “海量分类数据 + 有限检测数据” 的有效融合，突破检测模型的类别覆盖限制，且保持实时检测能力；
局限：对 COCO 中无对应边界框标注的类别（如服装、装备类 “墨镜”“泳裤”）检测效果极差（AP 接近 0），但对动物类等 COCO 有相关标注的细分类别（如 “小熊猫”“老虎”）表现较好（AP 50%+）。